# <center>计算机视觉HW1
## <center>三层神经网络分类器实现Fashion-MNIST图像分类实验报告
### <center>盖烈森    
### <center>23307130013@m.fudan.edu.cn


### 一、实验摘要
本实验基于纯 NumPy 手动搭建三层全连接神经网络分类器，在 Fashion-MNIST 服装数据集上完成 10 分类任务。实验全程未使用 PyTorch、TensorFlow 等自带自动微分的深度学习框架，自主实现了前向传播、反向传播与自动微分；完成了带动量的 SGD 优化器、余弦退火学习率衰减、带 L2 正则化的交叉熵损失函数、梯度裁剪等核心模块的开发；通过四维网格搜索完成了学习率、隐藏层大小、批量大小、L2 正则化强度共 81 组超参数的调优，根据验证集准确率自动保存最优模型权重并实现早停机制。最终最优模型在独立测试集上分类准确率达到90.08%，相比基线模型（不使用余弦退火、未将 batch size 加入超参搜索）提升 1.81 个百分点，达到了全连接网络在该任务上的优秀性能水平。实验完整完成了训练曲线可视化、第一层权重特征可视化、混淆矩阵分析与典型错例分析等全部要求，深入分析了全连接网络在图像分类任务上的优势与固有局限。

### 二、实验环境与数据集
#### 2.1 实验环境
| 类别 | 详细配置 |
| --- | --- |
| 硬件 | Apple M3 |
| 软件 | Python 3.13 |
| 依赖库 | NumPy进行矩阵运算、Matplotlib用于可视化、Scikit-learn用于混淆矩阵计算、TensorFlow仅用于官方数据集加载，不涉及数据处理和模型训练 |
| 随机种子 | 固定`np.random.seed(42)`，保证数据集划分和模型初始化的一致性 |

#### 2.2 实验可复现性说明
为确保实验结果完全可复现，本次实验采取了一系列严格措施。首先固定全局随机种子，确保每次运行的数据集划分和模型权重初始化完全一致，消除了随机因素对实验结果的影响。其次统一所有实验的硬件和软件环境，避免环境差异导致的结果波动。同时完整记录所有超参数组合的训练日志和验证结果，保存训练过程中所有最优模型的权重文件。最后提供完整的可运行代码，直接运行即可复现全部实验结果，包括数据加载、模型训练、超参数搜索和结果可视化的所有步骤。

#### 2.3 数据集说明
实验采用Fashion-MNIST官方原版数据集，该数据集是经典的图像分类基准数据集，包含70000张28×28的单通道灰度服装图像，共分为10个类别，具体为：T-shirt上衣、Trouser裤子、Pullover套头衫、Dress连衣裙、Coat外套、Sandal凉鞋、Shirt衬衫、Sneaker运动鞋、Bag包、Ankle boot短靴。数据集按照标准的机器学习划分规则进行划分，其中50000张图像作为训练集，用于模型权重更新与学习；10000张图像作为验证集，用于超参数调优与最优模型筛选；剩余10000张图像作为测试集，用于模型最终泛化性能评估，测试集在整个训练过程中完全不可见，确保评估结果的客观性。


### 三、实验原理
#### 3.1 三层神经网络结构
本实验搭建的三层MLP结构为输入层→隐藏层→输出层，定义输入样本为$x \in \mathbb{R}^{N \times 784}$，其中$N$为批量大小，784为28×28图像展平后的维度。第一层隐藏层的权重矩阵为$W_1 \in \mathbb{R}^{784 \times H}$，偏置向量为$b_1 \in \mathbb{R}^{1 \times H}$，其中$H$为隐藏层维度，可通过超参数自定义。输出层的权重矩阵为$W_2 \in \mathbb{R}^{H \times 10}$，偏置向量为$b_2 \in \mathbb{R}^{1 \times 10}$，10对应10个分类类别。模型通过线性变换与非线性激活函数的组合，学习从输入图像到类别概率的非线性映射关系。

#### 3.2 前向传播
前向传播是模型从输入到输出预测的计算过程，首先计算隐藏层的线性输出$z_1$，公式为：
$$ z_1 = xW_1 + b_1 $$
其中加法采用广播机制，将偏置向量$b_1$扩展为与$xW_1$相同的维度，应用于每个样本。

接着对线性输出应用非线性激活函数，本实验默认使用ReLU激活函数，公式为：
$$ a_1 = \text{ReLU}(z_1) = \max(0, z_1) $$
ReLU函数能够有效缓解梯度消失问题，同时计算简单，是深度学习中最常用的激活函数。

然后计算输出层的线性输出$z_2$，公式为：
$$ z_2 = a_1W_2 + b_2 $$

最后通过Softmax函数将输出层的线性输出归一化为概率分布，得到每个类别的预测概率。为了避免指数爆炸问题，本实验采用数值稳定的Softmax实现，公式为：
$$ a_{2,i,j} = \frac{\exp(z_{2,i,j} - \max_{k=1}^{10} z_{2,i,k})}{\sum_{k=1}^{10} \exp(z_{2,i,k} - \max_{k=1}^{10} z_{2,i,k})} $$
其中$a_{2,i,j}$表示第$i$个样本属于第$j$个类别的预测概率，通过减去每个样本的最大输出值，确保指数运算的数值稳定性。Softmax函数能够保证所有类别的预测概率之和为1，符合概率的基本性质，便于后续的损失计算和分类决策。

#### 3.3 反向传播与自动微分
反向传播基于链式法则，从损失函数出发，反向推导损失对每一层权重、偏置的梯度，用于参数更新。本实验手动实现了完整的梯度计算，首先定义真实标签为$y \in \mathbb{R}^{N \times 10}$，采用one-hot编码形式，即对于第$i$个样本，若其真实类别为$c$，则$y_{i,c}=1$，其余元素为0。

交叉熵损失对输出层线性输出$z_2$的梯度是反向传播的起点，经过推导可得：
$$ \frac{\partial L}{\partial z_2} = \frac{1}{N} (a_2 - y) $$
这是一个非常简洁的结果，也是Softmax函数与交叉熵损失函数配合使用的优势所在，避免了复杂的梯度计算。

接下来计算输出层权重$W_2$和偏置$b_2$的梯度，同时加入L2正则化项的梯度，公式为：
$$ \frac{\partial L}{\partial W_2} = a_1^T \frac{\partial L}{\partial z_2} + \lambda W_2 $$
$$ \frac{\partial L}{\partial b_2} = \sum_{i=1}^{N} \left( \frac{\partial L}{\partial z_2} \right)_i $$
其中$\lambda$为L2正则化强度，求和操作是对批量中所有样本的偏置梯度进行求和。

然后计算隐藏层激活输出$a_1$的梯度，根据链式法则：
$$ \frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} W_2^T $$

接着计算隐藏层线性输出$z_1$的梯度，需要乘以ReLU激活函数的梯度。ReLU函数的梯度为：
$$ \text{ReLU}'(z) = \begin{cases} 
1 & \text{if } z > 0 \\
0 & \text{otherwise}
\end{cases} $$
因此隐藏层线性输出的梯度为：
$$ \frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \text{ReLU}'(z_1) $$
其中$\odot$表示元素-wise乘法。

最后计算隐藏层权重$W_1$和偏置$b_1$的梯度，同样加入L2正则化项的梯度，公式为：
$$ \frac{\partial L}{\partial W_1} = x^T \frac{\partial L}{\partial z_1} + \lambda W_1 $$
$$ \frac{\partial L}{\partial b_1} = \sum_{i=1}^{N} \left( \frac{\partial L}{\partial z_1} \right)_i $$

手动实现反向传播能够深入理解神经网络的学习机制，避免了框架自动微分的黑箱问题，同时也为后续的优化和调试提供了便利。

#### 3.4 损失函数与正则化
本实验采用带L2正则化的多分类交叉熵损失函数，单样本的交叉熵损失公式为：
$$ L_i = -\sum_{j=1}^{10} y_{i,j} \log(a_{2,i,j}) $$
批量的平均交叉熵损失为：
$$ L_{\text{CE}} = \frac{1}{N} \sum_{i=1}^{N} L_i $$

为了防止模型过拟合，加入L2正则化项，对所有权重参数进行惩罚，公式为：
$$ L_{\text{reg}} = \frac{1}{2} \lambda \left( \|W_1\|_F^2 + \|W_2\|_F^2 \right) $$
其中$\|\cdot\|_F$表示Frobenius范数，即矩阵所有元素的平方和的平方根。

因此，最终的总损失函数为：
$$ L = L_{\text{CE}} + L_{\text{reg}} $$
L2正则化项通过惩罚权重的大小来防止模型过拟合，它能够使权重值尽可能小，避免模型过度依赖某些特定的特征，从而提高模型的泛化能力。正则化强度由超参数$\lambda$控制，$\lambda$越大，正则化效果越强，模型越不容易过拟合，但同时也可能导致模型欠拟合。

#### 3.5 优化器与学习率衰减
本实验采用带动量的SGD随机梯度下降方法，基于批量数据计算的梯度对模型参数进行迭代更新。定义参数$\theta$在第$t$步的梯度为$\nabla L(\theta_t)$，速度向量为$v_t$，则参数更新公式为：
$$ v_t = \gamma v_{t-1} - \eta_t \nabla L(\theta_t) $$
$$ \theta_{t+1} = \theta_t + v_t $$
其中$\eta_t$为第$t$步的学习率，$\gamma=0.9$为动量系数。动量项可以积累历史梯度信息，加速收敛并抑制训练过程中的震荡，使模型能够更快地收敛到最优解。

学习率调度采用余弦退火策略，相比传统的阶梯式衰减，余弦退火能够实现学习率的平滑下降，避免学习率突变导致的模型震荡，同时在训练后期保留微小的学习率用于精细调整参数。余弦退火的公式为：
$$ \eta_t = \eta_{\text{min}} + \frac{1}{2} (\eta_{\text{max}} - \eta_{\text{min}}) \left(1 + \cos\left(\frac{\pi \cdot t}{T}\right)\right) $$
其中$\eta_{\text{max}}$为初始最大学习率，$\eta_{\text{min}}$为最终最小学习率，本实验中设置$\eta_{\text{min}}=0$，$t$为当前epoch数，$T$为总训练epoch数。

#### 3.6 梯度裁剪
本次实验在参数更新前统一对所有参数的梯度执行全局梯度裁剪，确保训练过程稳定收敛。梯度裁剪基于梯度的L2范数进行约束，其核心思想是设定一个固定阈值，当梯度的L2范数超过该阈值时，对梯度进行等比例缩放，使其范数等于阈值；若梯度范数小于等于阈值，则保持梯度不变。设定梯度裁剪阈值为$\text{clip\_norm}$，本次实验将该阈值固定为1.0，该取值在多层全连接网络训练中被验证可有效抑制梯度爆炸且不破坏正常的参数更新方向。梯度裁剪的完整数学表达式为
$$g = \begin{cases} 
\displaystyle g \cdot \frac{\text{clip\_norm}}{\|g\|_2} & \|g\|_2 > \text{clip\_norm} \\
g & \|g\|_2 \le \text{clip\_norm}
\end{cases}$$
该操作仅对梯度的幅值进行约束，不会改变梯度的更新方向，因此不会影响模型的收敛方向，仅能限制参数更新的步长，避免因梯度过大导致模型参数剧烈波动、损失函数瞬间飙升甚至数值溢出。在本次三层神经网络的训练流程中，梯度裁剪执行于反向传播计算完成之后、优化器参数更新之前，对隐藏层权重梯度、隐藏层偏置梯度、输出层权重梯度、输出层偏置梯度全部统一进行范数计算与缩放处理，与带动量的SGD优化器、余弦退火学习率调度、L2正则化机制完全兼容。通过引入梯度裁剪机制，模型在使用较大初始学习率时依然能够保持训练稳定，有效避免了因梯度爆炸导致的训练中断问题，使模型能够收敛到更优的性能。

#### 3.7 早停机制
本实验中采用了早停的正则化方法，当验证集准确率连续多个epoch未提升时，提前终止训练，防止模型过拟合。定义验证集准确率在第$t$个epoch的值为$acc_t$，历史最高验证集准确率为$acc_{\text{best}}$，耐心值为$patience$，则早停条件为：
$$ \text{if } t - t_{\text{best}} > patience \text{ then stop training} $$
其中$t_{\text{best}}$为取得历史最高验证集准确率的epoch数。本实验中设置早停耐心值为15，即当验证集准确率连续15个epoch没有提升时，自动停止训练并保存最优模型权重。早停机制能够避免模型在训练集上过度拟合，同时节省训练时间，提高实验效率。

### 四、代码实现与模块说明
本实验代码采用模块化设计，共分为9个核心模块，各模块功能独立且耦合度低，既保证了代码的可读性与可维护性，也便于后续功能扩展与调优。

#### 4.1 数据加载与预处理模块
该模块通过函数`load_fashion_mnist(val_size=10000)`实现，核心完成Fashion-MNIST数据集的全流程预处理：调用`tensorflow.keras.datasets.fashion_mnist`仅加载官方原版数据集，后续处理完全手动实现，将28×28的二维图像张量展平为784维一维向量，通过`astype(np.float32)`统一数据类型，并除以255.0完成0-1归一化，消除像素值量级差异对模型训练的影响；基于随机索引抽样实现训练集与验证集的划分，通过`np.random.choice`随机选取`val_size`个样本作为验证集，剩余样本为训练集，测试集保持官方划分不变，默认将60000张训练图像划分为50000张训练集和10000张验证集，加载完成后打印各数据集维度信息，便于快速校验数据加载正确性。

#### 4.2 模型定义模块
该模块通过类`ThreeLayerMLP`实现，该类实现了三层全连接神经网络（输入层-隐藏层-输出层）的完整逻辑，支持自定义隐藏层大小、ReLU/Sigmoid两种激活函数切换，初始化时根据激活函数类型自动绑定对应的前向计算函数和梯度函数，并采用针对性的权重初始化策略——ReLU激活函数使用He初始化（`np.sqrt(2.0 / input_dim)`）以缓解梯度消失问题，Sigmoid激活函数使用Xavier初始化（`np.sqrt(1.0 / input_dim)`）保证输入输出方差一致，偏置项均初始化为全零矩阵；类中手动实现了前向传播流程，依次计算线性变换、激活函数输出和Softmax概率分布，同时基于链式法则手动推导并实现反向传播，包含L2正则项的梯度贡献，最终返回各参数的梯度字典；此外还提供了`load_weights()`方法，支持通过`pickle`导入训练好的最优模型权重，方便模型的测试和部署。

#### 4.3 核心工具函数模块
该模块通过函数`softmax()`、`cross_entropy_loss()`、`relu()`/`sigmoid()`及对应梯度函数实现，具体主要实现了Softmax激活、带L2正则的交叉熵损失、激活函数及其梯度的手动计算，不依赖任何框架。其中Softmax函数加入了数值稳定处理，通过减去每个样本的最大输出值来避免指数爆炸问题，提高了计算的稳定性；带L2正则的交叉熵损失函数同时衡量预测概率与真实标签的差距以及权重的复杂度，通过添加小常数`1e-10`避免对数运算的数值问题；ReLU和Sigmoid激活函数及其梯度函数均通过NumPy矩阵运算实现，确保计算效率与可解释性。

#### 4.4 SGD优化器模块
该模块通过类`SGDOptimizer`实现，主要实现了带动量的SGD随机梯度下降优化器。优化器采用惰性初始化策略，在第一次调用`step()`方法时才初始化与模型参数维度一致的速度向量，避免了不必要的内存占用；在每个训练步骤接收当前学习率，结合动量项累积历史梯度信息并更新模型参数，动量系数默认设为0.9，有效抑制了参数更新的震荡，实现了稳定高效的参数迭代。

#### 4.5 学习率调度模块
该模块通过函数`cosine_lr()`实现，实现了余弦退火学习率调度策略。函数接收当前epoch数、总epoch数和最大学习率，返回当前步骤的学习率，通过余弦函数将学习率从初始最大值平滑衰减至最小值（默认`1e-6`），公式为：
$$ \text{lr} = \text{lr}_{\text{min}} + 0.5 \times (\text{lr}_{\text{max}} - \text{lr}_{\text{min}}) \times (1 + \cos(\frac{\pi \times \text{epoch}}{\text{total\_epoch} - 1})) $$
该函数与优化器解耦，提高了代码的灵活性和可扩展性，可以方便地替换为其他学习率调度策略。

#### 4.6 训练循环模块
该模块通过函数`train()`实现，主要实现了完整的批量训练循环。每个epoch开始前通过`np.random.permutation`打乱训练集顺序，确保数据采样的随机性；随后按批量大小迭代训练集，依次完成前向传播、损失计算、反向传播、梯度裁剪和参数更新，其中梯度裁剪通过全局范数约束（默认`clip_norm=1.0`）防止梯度爆炸，学习率通过`cosine_lr()`函数动态调整；epoch结束后在训练集上计算损失和准确率，若启用验证集模式则同时在验证集上评估，并根据验证集准确率自动保存最优模型权重，若验证集准确率在`patience`（默认15）个epoch内未提升超过`min_delta`（默认`1e-4`）则触发早停机制；函数还支持全量数据训练模式，此时不进行验证集评估，训练结束后直接保存最后一轮模型权重，整个训练过程无需人工干预。

#### 4.7 超参数搜索模块
该模块通过函数`hyperparam_search()`实现，主要实现了四维网格搜索。搜索空间涵盖学习率（`[0.1, 0.15, 0.2]`）、隐藏层大小（`[512, 768, 1024]`）、批量大小（`[64, 128, 256]`）、L2正则化强度（`[5e-5, 1e-4, 0.001]`）的多组组合，每组参数独立训练`search_epochs`（默认5）个epoch并在验证集上评估，最终输出泛化性能最优的超参数组合；函数会自动记录所有超参数组合的验证集准确率，打印搜索进度，并在每组参数训练后删除临时权重文件，最终汇总并打印最优参数组合及对应的最高验证准确率。

#### 4.8 模型评估模块
该模块通过函数`evaluate()`实现，主要实现了计算模型在指定数据集上的分类准确率的计算评估，支持训练集、验证集、测试集的性能评估。分类准确率的计算公式为：
$$ \text{Accuracy} = \frac{\text{正确分类的样本数}}{\text{总样本数}} $$
函数通过前向传播得到预测概率，取概率最大的类别作为预测结果，计算预测结果与真实标签的平均准确率，实现简单高效的模型性能量化。

#### 4.9 可视化与分析模块
该模块通过函数`plot_history()`、`plot_confusion_matrix()`、`visualize_weights()`、`error_analysis()`实现，主要实现了训练Loss和Accuracy曲线绘制、混淆矩阵可视化、第一层权重图像化展示、分类错误样本可视化，为实验结果的分析提供了直观的支持。`plot_history()`函数绘制训练集（及验证集）的损失曲线和准确率曲线，便于观察模型收敛过程；`plot_confusion_matrix()`函数计算并绘制混淆矩阵，混淆矩阵$CM$的元素$CM_{i,j}$表示真实类别为$i$且被预测为类别$j$的样本数，通过混淆矩阵可以清晰地看到模型在各个类别上的分类表现以及主要的错误来源；`visualize_weights()`函数将第一层隐藏层权重矩阵恢复成28×28的图像尺寸并可视化，直观展示神经元学习到的特征；`error_analysis()`函数随机挑选（或全部展示）分类错误的样本，对比真实标签与预测标签并可视化样本图像，便于进行误差分析。

### 五、实验结果与分析
#### 5.1 超参数搜索结果
本次实验采用网格搜索进行超参数调优，共测试了81组超参数组合，搜索空间与最优结果如下：

| 超参数 | 搜索空间 | 最优取值 |
| --- | --- | --- |
| 初始最大学习率 | [0.1, 0.15, 0.2] | 0.1 |
| 隐藏层大小 | [512, 768, 1024] | 768 |
| 批量大小 | [64, 128, 256] | 128 |
| L2正则化强度 | [5e-5, 1e-4, 0.001] | 5e-5 |

最优超参数组合在5个epoch的快速验证中取得了 **88.59%** 的验证集准确率，下面通过控制变量方法展示各超参数对模型性能的影响。

#### 5.1.1 初始学习率对模型性能的影响
固定隐藏层维度768、批量大小128、L2正则化强度5e-5，仅改变初始学习率进行实验。当初始学习率为0.1时，模型取得了88.59%的最高验证集准确率，此时学习率适中，模型收敛速度与最终性能达到最佳平衡。当初始学习率提升至0.15时，验证集准确率下降至87.85%，这是因为学习率偏大导致前期震荡加剧，模型难以稳定收敛到最优解。当初始学习率进一步提升至0.2时，验证集准确率继续下降至87.64%，此时学习率过大，模型参数更新步长过大，在最优解附近来回震荡，无法收敛到最优解。学习率是影响模型性能的关键超参数，学习率过小会导致模型收敛速度慢，容易陷入局部最优；学习率过大会导致模型在最优解附近震荡，无法收敛。本次实验中0.1的初始学习率在收敛速度和最终精度之间取得了最佳平衡。

#### 5.1.2 隐藏层维度对模型性能的影响
固定初始学习率0.1、批量大小128、L2正则化强度5e-5，仅改变隐藏层维度进行实验。当隐藏层维度为512时，验证集准确率为88.39%，这是因为隐藏层维度较小，模型特征容量不足，无法充分捕捉复杂视觉特征，导致模型欠拟合。当隐藏层维度提升至768时，验证集准确率提升至88.59%，此时维度适中，特征容量与计算复杂度达到最佳平衡，模型能够充分学习到数据的特征。当隐藏层维度进一步提升至1024时，验证集准确率反而下降至88.38%，这是因为维度进一步提升出现了轻微的特征冗余，未带来性能提升反而增加了计算量，同时也增加了模型过拟合的风险。隐藏层维度决定了模型的特征表达能力，维度过小会导致模型欠拟合，无法学习到数据的复杂模式；维度过大会导致模型参数过多，容易过拟合且计算效率低下。本次实验中768维的隐藏层能够充分捕捉Fashion-MNIST数据集的特征，同时保持较高的计算效率。

#### 5.1.3 批量大小对模型性能的影响
固定初始学习率0.1、隐藏层维度768、L2正则化强度5e-5，仅改变批量大小进行实验。当批量大小为64时，验证集准确率为88.33%，这是因为批量过小，梯度噪声大，训练过程不稳定，模型参数更新方向波动较大。当批量大小提升至128时，验证集准确率提升至88.59%，此时批量适中，梯度估计准确且训练效率高，模型能够稳定收敛到最优解。当批量大小进一步提升至256时，验证集准确率下降至88.11%，这是因为批量过大，每个epoch的参数更新次数减少，模型收敛速度变慢，在相同的训练轮数下无法充分学习到数据的特征。批量大小影响梯度估计的准确性和训练效率，批量过小会导致梯度估计方差大，训练过程不稳定；批量过大会导致每个epoch的参数更新次数减少，模型收敛速度变慢。本次实验中128的批量大小在梯度估计准确性和训练效率之间取得了最佳平衡。

#### 5.1.4 L2正则化强度对模型性能的影响
固定初始学习率0.1、隐藏层维度768、批量大小128，仅改变L2正则化强度进行实验。当L2正则化强度为5e-5时，验证集准确率为88.59%，此时正则化强度适中，既能有效抑制过拟合，又不影响模型拟合能力。当L2正则化强度提升至1e-4时，验证集准确率下降至88.37%，这是因为正则化强度提升，开始轻微限制模型的拟合能力，导致模型无法充分学习到数据的特征。当L2正则化强度进一步提升至0.001时，验证集准确率显著下降至88.11%，此时正则化强度过大，模型出现明显的欠拟合，无法学习到数据的复杂模式。L2正则化通过惩罚权重的大小来防止模型过拟合，正则化强度过弱无法有效抑制过拟合；正则化强度过强会导致模型欠拟合，无法充分学习数据的特征。本次实验中5e-5的弱正则化既能防止过拟合，又能保证模型的表达能力。

#### 5.1.5 性能对比分析
本次实验结果与初始基线模型（未使用余弦退火、未将batch size加入超参搜索范围）相比，测试集准确率从88.27%提升至 **90.08%** ，提升了1.81个百分点；验证集最高准确率从89.33%提升至**90.26%**，提升了0.93个百分点。性能提升的主要原因包括更全面的超参数搜索，新增了批量大小的搜索，找到了更优的训练配置，使得模型在训练过程中能够更稳定地收敛。更优的学习率调度策略，余弦退火相比阶梯式衰减，能够在训练后期保留更合适的学习率，实现更精细的参数调整，使模型能够收敛到更优的解。更合理的正则化强度，将L2正则化强度从0.001降低到5e-5，避免了欠拟合问题，充分发挥了模型的拟合能力。此外，梯度裁剪机制的引入有效防止了训练过程中的梯度爆炸问题，提高了训练稳定性，使得模型能够在更大的学习率下稳定训练，进一步提升了模型的性能。

#### 5.2 训练过程曲线分析
本次实验最优模型共训练50个epoch，采用早停策略，耐心值为15。训练过程的Loss曲线与Accuracy曲线如下：

![](training_curve.png)

从Loss曲线中可以看出，训练集交叉熵损失整体呈持续下降趋势，从初始0.5871降至最终0.1253，说明模型在训练集上持续有效拟合，参数更新方向正确，无欠拟合问题。验证集损失整体同步下降，前期与训练损失高度贴合，仅在训练后期出现极其轻微的抬升，与训练损失的最大差距小于0.25，说明模型可能存在些许过拟合问题，这一点后续可能可以通过其他方法改进（见6.2改进建议）。训练过程中损失存在小幅波动，主要来自小批量训练的梯度噪声，余弦退火学习率的平滑下降有效抑制了大幅震荡。在训练后期，随着学习率逐渐降低，损失曲线变得更加平滑，说明模型进入了精细调整阶段，参数更新步长变小，模型在最优解附近进行微调。

Accuracy曲线显示，训练集准确率整体波动上升，最高达到 **97.96%** ；验证集准确率同步提升，最高达到 **90.26%** ，最终全量训练后测试集准确率 **90.08%** 与验证集最优值几乎一致，证明模型泛化能力极其优秀，在未见过的测试数据上表现稳定。训练集与验证集准确率的差距为7.70%，虽然可能存在一些过拟合问题，但我认为这是全连接网络在该任务上的正常差距，超参数选择是相对合理的，否则在测试集上的效果不会非常优秀地保持准确率在90%以上。在训练过程中，验证集准确率偶尔会出现小幅下降，这是由于小批量训练的随机性导致的，属于正常现象，早停机制能够有效避免模型在这些波动点停止训练，确保保存最优的模型权重。

#### 5.3 测试集性能与混淆矩阵分析
最优模型在独立测试集上的最终分类准确率为**90.08%**，混淆矩阵如下：

![](confusion_matrix.png)

基于混淆矩阵计算每个类别的分类准确率，模型在Trouser(裤子, 98.0%)、Bag(包, 97.2%)、Sandal(凉鞋, 96.7%)、Sneaker(运动鞋, 96.3%)、Ankle boot(短靴, 96.1%)上的分类准确率极高。这些类别视觉特征与其他类别差异显著，轮廓、结构独特，模型极易区分，分类效果优异。相比较而言，Shirt(衬衫, 73.3%)、Pullover(套头衫, 82.8%)、Coat(外套, 84.5%)、T-shirt(上衣, 85.2%)等衣物分类准确率较低。这些类别同属上衣类，视觉特征高度相似，均具备衣身、袖子的基础结构，轮廓与纹理差异小，导致模型难以区分。

主要错误来自上衣类的互相混淆，包括103个真实Shirt被错分为T-shirt、101个真实T-shirt被错分为Shirt、83个真实Pullover被错分为Coat、79个真实Coat被错分为Pullover，这类错误占总错误的70%以上，是模型性能的主要瓶颈。此外，还有少量其他类别的错误，如26个真实Sneaker被错分为Ankle boot、31个真实Ankle boot被错分为Sneaker，这是因为这两类都是鞋类，底部轮廓相似，模型容易混淆。这些错误分布与我的预期一致，进一步说明全连接网络在处理视觉特征高度相似的类别时存在局限性。

#### 5.4 第一层权重可视化分析
将训练好的第一层隐藏层权重矩阵恢复为28×28的图像，可视化结果如下：

![](weight_visualization_all.png)

图中每个小方块对应全连接网络第一层的一个神经元权重，重塑为与输入一致的28×28灰度图，直观展示了模型的特征学习效果。绝大多数神经元的权重图呈现出清晰的边缘、轮廓、纹理特征，部分权重图对应服装的竖向轮廓如裤子、连衣裙，部分对应横向衣身结构如上衣、T恤，部分对应局部纹理特征。这说明模型第一层成功学习到了输入图像的低级视觉特征，具备有效的特征提取能力，是模型分类性能的核心基础。仅少数神经元权重呈现无规律的噪声状，属于未被有效激活的冗余神经元，占比极低，说明768维的隐藏层维度与任务匹配度高，既无维度不足导致的欠拟合，也无严重的维度冗余。

通过观察权重图还可以发现，不同的神经元学习到了不同的特征，有的神经元对边缘敏感，有的对纹理敏感，有的对特定方向的线条敏感。这种特征的多样性使得模型能够从多个角度提取输入图像的特征，从而提高分类的准确性。同时，权重图的清晰度也反映了模型的训练效果，训练效果越好，权重图呈现的特征越清晰。

#### 5.5 错例分析
从测试集中随机抽取5个分类错误的典型样本，如下所示：

![](error_analysis.png)

这些错例完全印证了混淆矩阵的错误规律，第一个样本是真实Pullover被错分为Shirt，两者均为长袖上衣，全局轮廓、纹理几乎一致，模型无法区分领口、衣长等细微差异。第二个样本是真实Sneaker被错分为Ankle boot，两者均为鞋类，底部轮廓高度相似，模型仅能区分整体灰度分布，无法识别鞋帮高度的差异。第三个样本是真实Shirt被错分为Dress，该样本中的衬衫呈现宽松的长款形态，与连衣裙的全局轮廓相似，模型忽略了领口和袖子的特征。第四个样本是真实Dress被错分为Pullover，该样本中的连衣裙为短款设计，衣身部分与套头衫的轮廓高度相似，模型无法区分裙摆的细微特征。第五个样本是真实Pullover被错分为Coat，两者均为长袖上衣，整体轮廓高度相似，模型无法识别材质和厚度的差异。

这些错误充分反映了全连接网络的固有局限，全连接网络将图像展平为一维向量进行处理，丢失了图像的空间结构信息，无法有效捕捉局部细微特征差异。对于视觉特征高度相似的类别，全连接网络只能基于全局轮廓和灰度分布进行判断，难以区分细节差异。

### 六、实验结论
#### 6.1 核心结论
本次实验通过81组超参数网格搜索，确定了最优超参数组合：初始学习率0.1、隐藏层维度768、批量大小128、L2正则化系数5e-5。训练的全连接神经网络在Fashion-MNIST测试集上达到 **90.08%** 的分类准确率，超过了全连接网络在该任务上的平均性能水平。模型整体收敛稳定，虽存在些许过拟合问题，但还是成功学习到了服装图像的大部分有效视觉特征，对特征差异大的类别分类效果优异，泛化能力良好。模型的核心性能瓶颈仍然来自于全连接网络对相似类别局部特征的捕捉能力不足，以及数据集本身上衣类别的高视觉相似性。

#### 6.2 改进建议
基于本次实验的结果和分析，可以采取一系列措施进一步提升模型性能。首先是网络结构优化，替换全连接网络为卷积神经网络，利用卷积层的局部感受野特性，更好地捕捉图像的空间结构与细微特征差异，解决相似上衣类的混淆问题。其次是训练策略优化，引入数据增强措施，如随机水平翻转、微小裁剪、高斯噪声等，扩充训练数据的多样性，提升模型对相似类别的鲁棒性；尝试使用学习率预热策略，进一步稳定训练过程。此外，还可以优化正则化方法，加入Batch Normalization层与Dropout层，进一步加速模型收敛，抑制过拟合，提升训练稳定性。最后是超参数搜索优化，可能可以尝试使用贝叶斯搜索替代网格搜索，在相同计算资源下能够更高效地找到最优超参数组合；还可以尝试搜索动量系数、梯度裁剪阈值等更多超参数，进一步提升模型性能。

### 七、提交材料说明
#### 7.1 GitHub仓库地址
https://github.com/daomuyang/CV_HW1_Liesen_Gai

注：仓库README中包含完整的环境依赖说明、训练与测试脚本运行方式。

#### 7.2 最优模型权重下载地址
https://drive.google.com/drive/folders/1Z4h2Ulfl4KQsURjVt-o7Nlyc5zZ8XL8?dmr=1&ec=wgc-drive-%5Bmodule%5D-goto
中`final_best_model.pkl`文件

#### 7.3 实验报告附件
- 训练曲线：`training_curve.png`，包含训练集Loss/Accuracy曲线及验证集Loss/Accuracy曲线
- 混淆矩阵：`confusion_matrix.png`，展示模型在测试集上10个类别的分类混淆情况
- 权重可视化：`weight_visualization_all.png`，将第一层隐藏层所有神经元权重恢复为28×28图像的可视化结果
- 错例分析：`error_analysis.png`，随机展示5个测试集分类错误样本及其真实/预测标签对比
- 训练和测试脚本：`main.py`，直接运行即可自动完成数据加载与处理、超参数搜索、模型训练、测试评估及全部可视化结果生成的全过程
- 验证集最优模型权重：`val_best_model.pkl`，基于验证集准确率保存的最优模型权重
- 全量训练最优模型权重：`final_best_model.pkl`，使用训练集+验证集全量数据训练后的最终最优模型权重